In [1]:
import findspark
# Явно указываем путь к "родному" Spark, который зашит в этом Docker-образе
findspark.init('/usr/local/spark')

from pyspark.sql import SparkSession

def main():
    # 1. Инициализация SparkSession (точка входа в Spark)
    # Имя приложения будет отображаться в Spark UI
    spark = SparkSession.builder \
        .appName("WordCountScript") \
        .master("local[*]") \
        .getOrCreate()

    print(f"[*] Spark успешно запущен! Версия: {spark.version}")

    # 2. Создаем тестовые данные (RDD - Resilient Distributed Dataset)
    text_data = [
        "Hello Spark",
        "Hello Docker",
        "Spark is awesome and Docker is awesome too"
    ]

    rdd = spark.sparkContext.parallelize(text_data)
    print(f"[*] Количество партиций: {rdd.getNumPartitions()}")

    # 3. Выполняем классический MapReduce для подсчета слов
    word_counts = rdd.flatMap(lambda line: line.split(" ")) \
                     .map(lambda word: (word, 1)) \
                     .reduceByKey(lambda a, b: a + b)

    # 4. Собираем результаты на драйвер и выводим
    results = word_counts.collect()

    print("\n--- Результат WordCount ---")
    for word, count in results:
        print(f"'{word}': {count}")
    print("---------------------------\n")

    # 5. Остановка сессии для освобождения ресурсов (обязательная практика)
    spark.stop()
    print("[*] Spark сессия остановлена.")

if __name__ == "__main__":
    main()

[*] Spark успешно запущен! Версия: 3.5.0
[*] Количество партиций: 12

--- Результат WordCount ---
'and': 1
'too': 1
'Hello': 2
'is': 2
'awesome': 2
'Spark': 2
'Docker': 2
---------------------------

[*] Spark сессия остановлена.
